In [ ]:
!pip install accelerate tiktoken swanlab -q

In [ ]:
!pip uninstall -y -q tensorflow
!pip install -q tensorflow-cpu

这个环境补丁很重要！如果你用pytorch，并且是用kaggle的TPU来训练，不管你是用xla还是accelerate库，建议加上这个补丁，否则会报错。

In [ ]:
# TPU 环境补丁：必须最先执行
import os
os.environ["PJRT_DEVICE"] = "TPU"
os.environ["OMP_NUM_THREADS"] = "1"
if "TPU_PROCESS_ADDRESSES" in os.environ:
    del os.environ["TPU_PROCESS_ADDRESSES"]
if "CLOUD_TPU_TASK_ID" in os.environ:
    del os.environ["CLOUD_TPU_TASK_ID"]

# TPU memory optimizations
os.environ['XLA_TENSOR_ALLOCATOR_MAXSIZE'] = '100000000'

print("✅ TPU 环境补丁已设置")

下面的代码跟仓库的train_gpt2_tpu_accelerate.py一模一样，只是方便在kaggle上导入运行。

In [ ]:
%%writefile train_gpt2_tpu_accelerate.py
import os
# TPU memory optimizations
os.environ['XLA_TENSOR_ALLOCATOR_MAXSIZE'] = '100000000'

import math
import time
from dataclasses import dataclass, asdict
import numpy as np
import tiktoken
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn import functional as F
from accelerate import Accelerator, notebook_launcher
from accelerate.utils import set_seed


# ============================================================
# Global Config
# ============================================================

DATA_ROOT = "/kaggle/input/your-dataset/edu-fineweb10b"  # 改成你的语料的路径
OUT_DIR = "out"
BEST_CKPT_NAME = "best_ckpt.pt"
FINAL_CKPT_NAME = "final_ckpt.pt"

SEED = 1337

# 训练 batch 配置
TOTAL_BATCH_SIZE = 524288   # 全局 token batch
MICRO_BATCH_SIZE = 32       # 调小以避免 TPU HBM 溢出，32是一个合适的取值
SEQ_LEN = 1024

# 模型配置
MODEL_VOCAB_SIZE = 50304
N_LAYER = 12
N_HEAD = 12
N_EMBD = 768

# 优化器
WEIGHT_DECAY = 0.1
BETA1 = 0.9
BETA2 = 0.95
EPS = 1e-6
GRAD_CLIP = 1.0

# 学习率
MAX_LR = 6e-4
MIN_LR = MAX_LR * 0.1
WARMUP_STEPS = 715
MAX_STEPS = 19073        
LR_DECAY_STEPS = 19073   # 学习率衰减目标步数，和 MAX_STEPS 解耦，保证按总步数衰减

# 验证 / 保存
VAL_INTERVAL = 1000
VAL_STEPS = 20
SAVE_BEST_ONLY_WHEN_IMPROVED = True

# 采样
DO_SAMPLE = True
SAMPLE_INTERVAL = 1000
SAMPLE_NUM_RETURN_SEQS = 4
SAMPLE_MAX_LENGTH = 32
SAMPLE_TOPK = 50
SAMPLE_PROMPTS = [
    "Hello, I'm a language model,",
    "I'm a computer science student,",
    "Science is cool",
]

# TPU / Accelerate
TPU_NUM_PROCESSES = 8
MIXED_PRECISION = "bf16"   # "bf16" or "no"
USE_TORCH_COMPILE = False  # TPU 不建议开

# Attention
# TPU 上默认走手写 causal attention，不走 flash / fused / cuda sdpa 快路径
USE_SDPA = True

# wandb（实则swanlab）
WANDB_ENABLED = True
WANDB_PROJECT = "nanogpt-tpu"
WANDB_RUN_NAME = "nanogpt-kaggle-tpuv5e-8"
from kaggle_secrets import UserSecretsClient
# 这里导入你的swanlab的api，需要先在kaggle的add-ons，点击secrets设置
try:
    user_secrets = UserSecretsClient()
    SWANLAB_API_KEY = user_secrets.get_secret("SWANLAB_API_KEY")
except Exception as e:
    print(f"⚠️ 提取 Kaggle Secrets 失败: {e}")
    SWANLAB_API_KEY = None

# resume
RESUME_PATH = None  # 设置检查点路径会从那里继续训练，例如 "out/final_ckpt.pt"


# ============================================================
# Utils
# ============================================================

def get_lr(step: int) -> float:
    """学习率调度：先线性 warmup 到 MAX_LR，在 LR_DECAY_STEPS 步数内余弦衰减到 MIN_LR，之后保持 MIN_LR"""
    if step < WARMUP_STEPS:
        return MAX_LR * (step + 1) / WARMUP_STEPS
    if step > LR_DECAY_STEPS:
        return MIN_LR

    decay_ratio = (step - WARMUP_STEPS) / (LR_DECAY_STEPS - WARMUP_STEPS)
    decay_ratio = min(max(decay_ratio, 0.0), 1.0)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return MIN_LR + coeff * (MAX_LR - MIN_LR)


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def load_tokens(filename):
    arr = np.load(filename).astype(np.int32)
    return torch.tensor(arr, dtype=torch.long)


# ============================================================
# Data Loader
# ============================================================

class StatefulShardLoader:
    """
    顺序读取 shard，并按多进程切分 batch。
    这里保存的是“全局同步状态”：
      - current_shard
      - batch_idx_in_shard
    这样 rank0 保存的 checkpoint 可以让所有 rank 正确 resume。
    """

    def __init__(self, B, T, process_rank, num_processes, split, data_root=DATA_ROOT, master_process=True):
        self.B = B
        self.T = T
        self.process_rank = process_rank
        self.num_processes = num_processes
        self.split = split
        self.data_root = data_root

        assert split in {"train", "val"}

        shards = os.listdir(data_root)
        shards = [s for s in shards if split in s]
        shards = sorted(shards)
        shards = [os.path.join(data_root, s) for s in shards]
        assert len(shards) > 0, f"no shards found for split={split} in {data_root}"

        self.shards = shards
        self.global_stride = self.B * self.T * self.num_processes

        if master_process:
            print(f"[{split}] found {len(shards)} shards", flush=True)

        self.reset()

    def reset(self):
        self.current_shard = 0
        self.batch_idx_in_shard = 0
        self.tokens = load_tokens(self.shards[self.current_shard])

    def _advance_shard(self):
        self.current_shard = (self.current_shard + 1) % len(self.shards)
        self.batch_idx_in_shard = 0
        self.tokens = load_tokens(self.shards[self.current_shard])

    def _has_enough_tokens_for_synced_batch(self):
        # 第 n 个同步 batch 需要的最大 token 索引上界：
        # (n+1) * global_stride + 1
        needed = (self.batch_idx_in_shard + 1) * self.global_stride + 1
        return needed <= len(self.tokens)

    def next_batch(self):
        while not self._has_enough_tokens_for_synced_batch():
            self._advance_shard()

        start = self.batch_idx_in_shard * self.global_stride + self.process_rank * self.B * self.T
        end = start + self.B * self.T + 1

        buf = self.tokens[start:end]
        x = buf[:-1].view(self.B, self.T)
        y = buf[1:].view(self.B, self.T)

        self.batch_idx_in_shard += 1
        return x, y

    def state_dict(self):
        return {
            "current_shard": self.current_shard,
            "batch_idx_in_shard": self.batch_idx_in_shard,
        }

    def load_state_dict(self, state):
        self.current_shard = int(state["current_shard"])
        self.batch_idx_in_shard = int(state["batch_idx_in_shard"])
        self.tokens = load_tokens(self.shards[self.current_shard])


# ============================================================
# Model
# ============================================================

def rotate_half(x):
    """将向量前半和后半交换并取反，等价于旋转 90 度"""
    x1 = x[..., :x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]
    return torch.cat([-x2, x1], dim=-1)


class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0

        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1

        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.use_sdpa = config.use_sdpa
        self.head_dim = config.n_embd // config.n_head
        self.scale = 1.0 / math.sqrt(self.head_dim)

        # QK-Norm：在 head_dim 维度上归一化，防止 attention logit 爆炸
        self.q_norm = nn.LayerNorm(self.head_dim)
        self.k_norm = nn.LayerNorm(self.head_dim)

        # RoPE：预计算 cos/sin 缓存
        freqs = 1.0 / (10000 ** (torch.arange(0, self.head_dim, 2) / self.head_dim))
        positions = torch.arange(config.block_size)
        angles = positions.unsqueeze(1) * freqs.unsqueeze(0)  # (block_size, head_dim/2)
        cos = angles.cos()
        sin = angles.sin()
        # 扩展到 head_dim 维度（前后半相同）
        self.register_buffer("cos_cached", torch.cat([cos, cos], dim=-1))  # (block_size, head_dim)
        self.register_buffer("sin_cached", torch.cat([sin, sin], dim=-1))  # (block_size, head_dim)

        mask = torch.tril(torch.ones(config.block_size, config.block_size, dtype=torch.bool))
        self.register_buffer(
            "causal_mask",
            mask.view(1, 1, config.block_size, config.block_size),
            persistent=False,
        )

    def forward(self, x):
        B, T, C = x.size()

        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)

        head_dim = C // self.n_head
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)

        # QK-Norm
        q = self.q_norm(q.float()).to(q.dtype)
        k = self.k_norm(k.float()).to(k.dtype)

        # RoPE + Attention（全路径 fp32）
        cos = self.cos_cached[:T].unsqueeze(0).unsqueeze(0)  # (1, 1, T, head_dim) 
        sin = self.sin_cached[:T].unsqueeze(0).unsqueeze(0)  
        q = q.float()
        k = k.float()
        v = v.float()
        q = q * cos + rotate_half(q) * sin
        k = k * cos + rotate_half(k) * sin

        if self.use_sdpa and x.device.type != "xla":
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        else:
            att = (q * self.scale) @ k.transpose(-2, -1)  # (B, nh, T, T) fp32
            mask_slice = self.causal_mask[:, :, :T, :T]
            att = att.masked_fill(~mask_slice, -1e4)
            att = F.softmax(att, dim=-1)
            y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y) 
        return y


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1

    def forward(self, x):
        x = self.c_fc(x)
        x = F.relu(x) ** 2  # ReLU²：稀疏 + 更强非线性
        return self.c_proj(x)


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        # LayerNorm 强制 fp32，防止 bf16 下方差 catastrophic cancellation 导致 NaN
        x = x + self.attn(self.ln_1(x.float()).to(x.dtype))
        x = x + self.mlp(self.ln_2(x.float()).to(x.dtype))
        return x


@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50304
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    use_sdpa: bool = False


class GPT(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.gradient_checkpointing = False

        self.transformer = nn.ModuleDict({
            "wte": nn.Embedding(config.vocab_size, config.n_embd),
            "h": nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            "ln_f": nn.LayerNorm(config.n_embd),
        })
        # 不设 lm_head，forward 中直接用 F.linear(x, wte.weight)
        # 从根本上避免 weight tying 被 accelerator.prepare() 破坏的问题
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            if hasattr(module, "NANOGPT_SCALE_INIT"):
                # 残差连接前的投影层初始化为零，初期残差流为恒等映射
                torch.nn.init.zeros_(module.weight)
            else:
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        assert T <= self.config.block_size, f"T={T} > block_size={self.config.block_size}"

        tok_emb = self.transformer.wte(idx)
        x = tok_emb

        try:
            from torch_xla.utils.checkpoint import checkpoint as xla_checkpoint
            HAS_XLA_CHECKPOINT = True
        except ImportError:
            HAS_XLA_CHECKPOINT = False

        # 使用梯度检查点（gradient checkpointing）来节省内存
        for block in self.transformer.h:
            if self.gradient_checkpointing and self.training:
                if x.device.type == "xla" and HAS_XLA_CHECKPOINT:
                    x = xla_checkpoint(block, x)
                else:
                    x = torch.utils.checkpoint.checkpoint(block, x, use_reentrant=False)
            else:
                x = block(x)

        x = self.transformer.ln_f(x.float()).to(x.dtype)
        logits = F.linear(x, self.transformer.wte.weight)

        loss = None
        if targets is not None:
            # logits 转 fp32 再算 cross_entropy，防止 bf16 下 log_softmax 溢出为 NaN
            loss = F.cross_entropy(
                logits.float().view(-1, logits.size(-1)),
                targets.view(-1)
            )
        return logits, loss

    def configure_optimizers(self, weight_decay, learning_rate, betas=(0.9, 0.95), eps=1e-8, master_process=True):
        param_dict = {pn: p for pn, p in self.named_parameters() if p.requires_grad}

        decay_params = [p for _, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for _, p in param_dict.items() if p.dim() < 2]

        optim_groups = [
            {"params": decay_params, "weight_decay": weight_decay},
            {"params": nodecay_params, "weight_decay": 0.0},
        ]

        if master_process:
            num_decay_params = sum(p.numel() for p in decay_params)
            num_nodecay_params = sum(p.numel() for p in nodecay_params)
            print(f"num decayed tensors: {len(decay_params)} | params: {num_decay_params:,}", flush=True)
            print(f"num non-decayed tensors: {len(nodecay_params)} | params: {num_nodecay_params:,}", flush=True)

        optimizer = torch.optim.AdamW(
            optim_groups,
            lr=learning_rate,
            betas=betas,
            eps=eps,
        )
        return optimizer


# ============================================================
# Eval / Sample / Checkpoint
# ============================================================

@torch.no_grad()
def evaluate(model, val_loader, accelerator, val_steps, use_xla_sync=False):
    model.eval()
    val_loader.reset()

    losses = []

    for _ in range(val_steps):
        x, y = val_loader.next_batch()
        x = x.to(accelerator.device)
        y = y.to(accelerator.device)

        with accelerator.autocast():
            _, loss = model(x, y)

        # TPU 上每步强制同步：立即执行当前步的计算图，物化 loss 值，
        # 释放 logits (B, T, vocab_size) 等大中间张量，防止多步累积导致 OOM
        if use_xla_sync:
            import torch_xla
            torch_xla.sync()

        losses.append(loss.detach())

    local_mean = torch.stack(losses).float().mean()
    global_mean = accelerator.gather(local_mean).mean().item()

    model.train()
    return global_mean


@torch.no_grad()
def generate_samples(model, enc, accelerator, step):
    model.eval()

    if accelerator.process_index == 0:
        print(f"\n===== sample @ step {step} =====", flush=True)

    for prompt in SAMPLE_PROMPTS:
        tokens = enc.encode(prompt)
        xgen = torch.tensor(tokens, dtype=torch.long, device=accelerator.device).unsqueeze(0)
        xgen = xgen.repeat(SAMPLE_NUM_RETURN_SEQS, 1)

        while xgen.size(1) < SAMPLE_MAX_LENGTH:
            with accelerator.autocast():
                logits, _ = model(xgen)

            logits = logits[:, -1, :]
            probs = F.softmax(logits.float(), dim=-1)
            topk_probs, topk_indices = torch.topk(probs, SAMPLE_TOPK, dim=-1)
            ix = torch.multinomial(topk_probs, 1)
            xcol = torch.gather(topk_indices, -1, ix)
            xgen = torch.cat((xgen, xcol), dim=1)

        if accelerator.process_index == 0:
            print(f"\n[prompt] {prompt}", flush=True)
            for i in range(SAMPLE_NUM_RETURN_SEQS):
                out = enc.decode(xgen[i].tolist())
                print(f"  sample {i}: {out}", flush=True)

    if accelerator.process_index == 0:
        print("================================\n", flush=True)

    model.train()


def save_checkpoint(
    accelerator,
    model,
    optimizer,
    train_loader,
    val_loader,
    step,
    best_val_loss,
    model_config,
    save_path,
):
    accelerator.wait_for_everyone()

    state_dict = accelerator.get_state_dict(model)
    opt_state = optimizer.state_dict()

    ckpt = {
        "model": state_dict,
        "optimizer": opt_state,
        "step": step,
        "best_val_loss": best_val_loss,
        "model_config": asdict(model_config),
        "train_loader_state": train_loader.state_dict(),
        "val_loader_state": val_loader.state_dict(),
        "global_config": {
            "TOTAL_BATCH_SIZE": TOTAL_BATCH_SIZE,
            "MICRO_BATCH_SIZE": MICRO_BATCH_SIZE,
            "SEQ_LEN": SEQ_LEN,
            "MAX_LR": MAX_LR,
            "MIN_LR": MIN_LR,
            "WARMUP_STEPS": WARMUP_STEPS,
            "MAX_STEPS": MAX_STEPS,
            "LR_DECAY_STEPS": LR_DECAY_STEPS,
            "WEIGHT_DECAY": WEIGHT_DECAY,
            "BETA1": BETA1,
            "BETA2": BETA2,
            "EPS": EPS,
            "GRAD_CLIP": GRAD_CLIP,
            "MIXED_PRECISION": MIXED_PRECISION,
        },
    }

    accelerator.save(ckpt, save_path)
    accelerator.wait_for_everyone()


# ============================================================
# Train
# ============================================================

def train_worker():
    import builtins

    ensure_dir(OUT_DIR)

    # 梯度累积步数必须在 Accelerator 构造前算出，作为参数传入
    # accumulate() 上下文管理器依赖此值决定何时做梯度同步
    world_size = TPU_NUM_PROCESSES
    assert TOTAL_BATCH_SIZE % (MICRO_BATCH_SIZE * SEQ_LEN * world_size) == 0, (
        f"TOTAL_BATCH_SIZE={TOTAL_BATCH_SIZE} must be divisible by "
        f"MICRO_BATCH_SIZE*SEQ_LEN*world_size = {MICRO_BATCH_SIZE * SEQ_LEN * world_size}"
    )
    grad_accum_steps = TOTAL_BATCH_SIZE // (MICRO_BATCH_SIZE * SEQ_LEN * world_size)

    accelerator = Accelerator(
        mixed_precision=MIXED_PRECISION,
        gradient_accumulation_steps=grad_accum_steps,
    )

    use_xla_sync = accelerator.device.type == "xla"

    def tpu_print(*args, **kwargs):
        if accelerator.process_index == 0:
            kwargs["flush"] = True
            builtins.print(*args, **kwargs)

    process_seed = SEED + accelerator.process_index
    set_seed(process_seed)
    device = accelerator.device

    tpu_print(f"device = {device}")
    tpu_print(f"world_size = {world_size}")
    tpu_print(f"process_index = {accelerator.process_index}")
    tpu_print(f"mixed_precision = {MIXED_PRECISION}")
    tpu_print(f"grad_accum_steps(local) = {grad_accum_steps}")
    tpu_print(f"total batch size(tokens) = {TOTAL_BATCH_SIZE}")

    wandb = None
    if WANDB_ENABLED and accelerator.process_index == 0:
        try:
            import swanlab as wandb

            if SWANLAB_API_KEY:
                wandb.login(SWANLAB_API_KEY)

            wandb.init(
                project=WANDB_PROJECT,
                name=WANDB_RUN_NAME,
                config={
                    "data_root": DATA_ROOT,
                    "total_batch_size": TOTAL_BATCH_SIZE,
                    "micro_batch_size": MICRO_BATCH_SIZE,
                    "seq_len": SEQ_LEN,
                    "grad_accum_steps_local": grad_accum_steps,
                    "world_size": world_size,
                    "n_layer": N_LAYER,
                    "n_head": N_HEAD,
                    "n_embd": N_EMBD,
                    "vocab_size": MODEL_VOCAB_SIZE,
                    "weight_decay": WEIGHT_DECAY,
                    "beta1": BETA1,
                    "beta2": BETA2,
                    "eps": EPS,
                    "max_lr": MAX_LR,
                    "min_lr": MIN_LR,
                    "warmup_steps": WARMUP_STEPS,
                    "max_steps": MAX_STEPS,
                    "lr_decay_steps": LR_DECAY_STEPS,
                    "mixed_precision": MIXED_PRECISION,
                    "use_sdpa": USE_SDPA,
                    "resume_path": RESUME_PATH,
                },
            )
            tpu_print("wandb initialized")
        except Exception as e:
            tpu_print(f"wandb init failed, continue without wandb: {e}")
            wandb = None

    accelerator.wait_for_everyone()

    # -------------------------
    # Model / Optimizer / Resume
    # -------------------------

    start_step = 0
    best_val_loss = float("inf")
    checkpoint = None

    model_config = GPTConfig(
        block_size=SEQ_LEN,
        vocab_size=MODEL_VOCAB_SIZE,
        n_layer=N_LAYER,
        n_head=N_HEAD,
        n_embd=N_EMBD,
        use_sdpa=USE_SDPA,
    )

    if RESUME_PATH is not None:
        tpu_print(f"resuming from {RESUME_PATH}")
        checkpoint = torch.load(RESUME_PATH, map_location="cpu")

        if "model_config" in checkpoint:
            model_config = GPTConfig(**checkpoint["model_config"])

        model = GPT(model_config)
        state_dict = checkpoint["model"]

        # 清理可能的前缀（torch.compile 会加 _orig_mod.，distributed 会加 module.）
        unwanted_prefixes = ['_orig_mod.', 'module.']
        for prefix in unwanted_prefixes:
            for k in list(state_dict.keys()):
                if k.startswith(prefix):
                    state_dict[k[len(prefix):]] = state_dict.pop(k)

        missing, unexpected = model.load_state_dict(state_dict, strict=False)
        if missing:
            tpu_print(f"⚠️ load_state_dict 缺失的 key: {missing}")
        if unexpected:
            tpu_print(f"⚠️ load_state_dict 多余的 key: {unexpected}")

        # 确保所有参数为 fp32（checkpoint 可能是 bf16 保存的）
        for name, param in model.named_parameters():
            if param.dtype != torch.float32:
                tpu_print(f"⚠️ {name} dtype={param.dtype}，转为 fp32")
                param.data = param.data.float()

        # 验证权重是否真的加载成功
        first_param_name, first_param = next(model.named_parameters())
        tpu_print(f"验证加载: {first_param_name}[:5] = {first_param.data.view(-1)[:5].tolist()}")

        start_step = int(checkpoint["step"])
        best_val_loss = float(checkpoint["best_val_loss"])
    else:
        tpu_print("initializing model from scratch")
        model = GPT(model_config)

    model.gradient_checkpointing = True

    if USE_TORCH_COMPILE:
        tpu_print("warning: torch.compile is disabled on TPU in practice; keeping it off is recommended")
        model = torch.compile(model)

    optimizer = model.configure_optimizers(
        weight_decay=WEIGHT_DECAY,
        learning_rate=MAX_LR,
        betas=(BETA1, BETA2),
        eps=EPS,
        master_process=(accelerator.process_index == 0),
    )

    model, optimizer = accelerator.prepare(model, optimizer)

    # 无 lm_head，不存在 weight tying 问题，无需诊断

    # 强制 XLA 同步：确保从 CPU 加载的权重真正物化到 TPU 内存
    if use_xla_sync:
        import torch_xla.core.xla_model as xm
        xm.mark_step()

    # optimizer 状态必须在 prepare 之后加载
    if checkpoint is not None and "optimizer" in checkpoint:
        tpu_print("loading optimizer state...")
        optimizer.load_state_dict(checkpoint["optimizer"])
        if use_xla_sync:
            xm.mark_step()

    # -------------------------
    # Data
    # -------------------------

    train_loader = StatefulShardLoader(
        B=MICRO_BATCH_SIZE,
        T=SEQ_LEN,
        process_rank=accelerator.process_index,
        num_processes=world_size,
        split="train",
        data_root=DATA_ROOT,
        master_process=(accelerator.process_index == 0),
    )
    val_loader = StatefulShardLoader(
        B=MICRO_BATCH_SIZE,
        T=SEQ_LEN,
        process_rank=accelerator.process_index,
        num_processes=world_size,
        split="val",
        data_root=DATA_ROOT,
        master_process=(accelerator.process_index == 0),
    )

    if checkpoint is not None:
        if "train_loader_state" in checkpoint:
            train_loader.load_state_dict(checkpoint["train_loader_state"])
        if "val_loader_state" in checkpoint:
            val_loader.load_state_dict(checkpoint["val_loader_state"])

    enc = tiktoken.get_encoding("gpt2")

    # -------------------------
    # Train Loop
    # -------------------------

    raw_model = accelerator.unwrap_model(model)
    step = start_step

    tpu_print(f"start_step = {start_step}")
    tpu_print(f"best_val_loss = {best_val_loss}")

    while step < MAX_STEPS:
        t0 = time.time()
        last_step = (step == MAX_STEPS - 1)

        # eval
        if step % VAL_INTERVAL == 0 or last_step:
            val_loss = evaluate(
                model=model,
                val_loader=val_loader,
                accelerator=accelerator,
                val_steps=VAL_STEPS,
                use_xla_sync=use_xla_sync,
            )

            if accelerator.process_index == 0:
                tpu_print(f"step {step} | val_loss {val_loss:.4f}")
                if wandb is not None:
                    wandb.log({"val/loss": val_loss, "step": step}, step=step)

            improved = val_loss < best_val_loss
            if improved:
                best_val_loss = val_loss

            if step > 0 and improved and SAVE_BEST_ONLY_WHEN_IMPROVED:
                best_path = os.path.join(OUT_DIR, BEST_CKPT_NAME)
                tpu_print(f"saving best checkpoint to {best_path}")
                save_checkpoint(
                    accelerator=accelerator,
                    model=model,
                    optimizer=optimizer,
                    train_loader=train_loader,
                    val_loader=val_loader,
                    step=step,
                    best_val_loss=best_val_loss,
                    model_config=raw_model.config,
                    save_path=best_path,
                )

        # sample
        if DO_SAMPLE and ((step > 0 and step % SAMPLE_INTERVAL == 0) or last_step):
            accelerator.wait_for_everyone()
            # TPU 死锁大坑修复：XLA 的计算图要求多卡同步。所有设备都必须参与 generate 运算进行占位，绝不能 `if rank == 0:` 单独执行
            generate_samples(raw_model, enc, accelerator, step)
            accelerator.wait_for_everyone()

        # train one optimizer step
        # 使用 accumulate 上下文管理器：前 N-1 步跳过梯度同步（all-reduce），最后一步才同步
        # 比手动调用 accelerator.backward() 少 N-1 次 all-reduce 通信
        model.train()
        optimizer.zero_grad(set_to_none=True)

        loss_accum = torch.zeros((), device=device)

        lr = get_lr(step)
        for group in optimizer.param_groups:
            group["lr"] = lr

        with accelerator.accumulate(model):
            for micro_step in range(grad_accum_steps):
                x, y = train_loader.next_batch()
                x = x.to(device)
                y = y.to(device)

                with accelerator.autocast():
                    _, loss = model(x, y)

                loss_accum += loss.detach() / grad_accum_steps
                # accumulate 上下文会自动处理 loss 缩放和梯度同步时机
                accelerator.backward(loss)

            grad_norm = None
            if GRAD_CLIP is not None and GRAD_CLIP > 0:
                grad_norm = accelerator.clip_grad_norm_(model.parameters(), GRAD_CLIP)

            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        if use_xla_sync:
            import torch_xla
            torch_xla.sync()

        global_loss = accelerator.gather(loss_accum).mean().item()
        global_grad_norm = None
        if grad_norm is not None:
            grad_norm_tensor = grad_norm.detach() if torch.is_tensor(grad_norm) else torch.tensor(float(grad_norm), device=device)
            global_grad_norm = accelerator.gather(grad_norm_tensor).mean().item()

        t1 = time.time()
        dt = t1 - t0
        tokens_processed = MICRO_BATCH_SIZE * SEQ_LEN * grad_accum_steps * world_size
        tok_per_sec = tokens_processed / max(dt, 1e-6)

        if accelerator.process_index == 0:
            msg = (
                f"step {step:5d} | "
                f"loss {global_loss:.6f} | "
                f"lr {lr:.4e} | "
                f"grad_norm {global_grad_norm if global_grad_norm is not None else float('nan'):.4f} | "
                f"dt {dt * 1000:.2f}ms | "
                f"tok/s {tok_per_sec:.2f}"
            )
            tpu_print(msg)

            if wandb is not None:
                wandb.log(
                    {
                        "train/loss": global_loss,
                        "train/lr": lr,
                        "train/grad_norm": global_grad_norm if global_grad_norm is not None else float("nan"),
                        "perf/tok_per_sec": tok_per_sec,
                        "perf/step_time_ms": dt * 1000.0,
                        "step": step,
                    },
                    step=step,
                )

        step += 1

    # -------------------------
    # Final Checkpoint
    # -------------------------

    final_path = os.path.join(OUT_DIR, FINAL_CKPT_NAME)
    tpu_print(f"saving final checkpoint to {final_path}")
    save_checkpoint(
        accelerator=accelerator,
        model=model,
        optimizer=optimizer,
        train_loader=train_loader,
        val_loader=val_loader,
        step=step,
        best_val_loss=best_val_loss,
        model_config=raw_model.config,
        save_path=final_path,
    )

    if accelerator.process_index == 0 and wandb is not None:
        wandb.finish()

    accelerator.wait_for_everyone()
    tpu_print("training done")


# ============================================================
# Launch
# ============================================================

def launch():
    notebook_launcher(train_worker, num_processes=TPU_NUM_PROCESSES)


if __name__ == "__main__":
    launch()

In [ ]:
from train_gpt2_tpu_accelerate import launch
launch()